In [4]:
import os
import re
import subprocess

import whisper
import yt_dlp


class ReadMateSTT:
    def __init__(self, model_size="medium"):
        print(f">>> Whisper 모델({model_size}) 로딩 중...")
        self.model = whisper.load_model(model_size)
        self.temp_dir = "temp_audio"
        os.makedirs(self.temp_dir, exist_ok=True)

    def extract_from_youtube(self, url):
        """유튜브 URL에서 오디오만 추출하여 wav 파일로 저장"""
        output_path = os.path.join(self.temp_dir, "target_audio")
        
        ydl_opts = {
            'format': 'bestaudio/best',
            'postprocessors': [{
                'key': 'FFmpegExtractAudio',
                'preferredcodec': 'wav',
            }],
            'outtmpl': output_path,
            'quiet': True,
        }

        print(f">>> 유튜브 오디오 추출 시작: {url}")
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])
        
        return f"{output_path}.wav"

    def preprocess_audio(self, input_path):
        """Whisper 최적 포맷으로 오디오 전처리 (16kHz, 모노, 음량 정규화)"""
        output_path = input_path.replace(".wav", "_processed.wav")
        print(">>> 오디오 전처리 중...")
        subprocess.run([
            "ffmpeg", "-i", input_path,
            "-ar", "16000",
            "-ac", "1",
            "-af", "loudnorm",
            output_path, "-y", "-q:a", "0"
        ], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        return output_path

    def clean_text(self, text):
        """추임새 및 중복 제거"""
        # 단독 추임새만 제거 (의미 있는 단어 보호)
        cleaned = re.sub(r"\b(어+|음+|아+|그+|저+|에+)\b", "", text)
        return re.sub(r"\s+", " ", cleaned).strip()

    def run_pipeline(self, youtube_url):
        """추출부터 정제까지 한 번에 실행"""
        audio_file = None
        processed_file = None
        try:
            # 1. 유튜브에서 wav 추출
            audio_file = self.extract_from_youtube(youtube_url)
            
            # 2. 오디오 전처리
            processed_file = self.preprocess_audio(audio_file)

            # 3. STT 변환
            print(">>> 텍스트 변환 시작...")
            result = self.model.transcribe(
                processed_file,
                fp16=False,
                language="ko",
                beam_size=5,
                best_of=5,
                temperature=0.0,
                condition_on_previous_text=False,
                no_speech_threshold=0.6,
                compression_ratio_threshold=2.4,
            )
            
            # 4. 텍스트 정제
            raw_text = result['text']
            cleaned_text = self.clean_text(raw_text)
            
            print("\n" + "="*100)
            print("[최종 추출 결과]")
            print(cleaned_text[:300] + "...")
            print("="*100)
            
            # 파일로 저장
            with open("moonbns_lecture_output.txt", "w", encoding="utf-8") as f:
                f.write(cleaned_text)
                
            return cleaned_text

        except Exception as e:
            print(f"!!! 오류 발생: {e}")
        finally:
            # 임시 파일 삭제
            for path in [audio_file, processed_file]:
                if path and os.path.exists(path):
                    os.remove(path)

# --- 실제 실행부 ---
if __name__ == "__main__":
    stt_worker = ReadMateSTT(model_size="medium")
    
    test_url = "https://youtu.be/VmFFlK03WHc?si=kPjOz_fth2iVzcKL"
    
    stt_worker.run_pipeline(test_url)

>>> Whisper 모델(medium) 로딩 중...
>>> 유튜브 오디오 추출 시작: https://youtu.be/VmFFlK03WHc?si=kPjOz_fth2iVzcKL


>>> 오디오 전처리 중...                                         
>>> 텍스트 변환 시작...

[최종 추출 결과]
우리가 드디어 해방이 됐어. 언제? 1945년 8월 15일 날. 해방이 됐어. 이게 어떤 의미를 갖느냐? 자 이렇게 지도를 본다고 했을 때 우리가 해방이 됐던 거는 일본이 무조건 항복을 했기 때문이야. 우리나라는 그때 오피셜하게 일본의 식민지였던 거지. 식민지를 어떻게 할 거냐를 두고서 이채세계대전의 승정국들이 걔네들 편하라고 38선을 거었단 말이야. 다음에 위에는 누가 들어와? 소련. 밑에는 누가 들어와? 미국. 어떤 차이가 있는지 말해줄게. 소련은 김일선 단독정보를 인정했어. 왜냐면은 공산주의 정부가 들어서는군요. 알겠습니다. ...


In [ ]:
from services.stt import ReadMateSTT

stt_worker = ReadMateSTT(model_size="medium")
test_url = "https://youtu.be/VmFFlK03WHc?si=kPjOz_fth2iVzcKL"
stt_worker.run_pipeline(test_url)